# Color

In [ ]:
import numpy as np
import trimesh
from scipy.spatial import cKDTree

# ============================================================
# COLOR TRANSFER
# ============================================================

def transfer_vertex_colors(original_mesh, simplified_mesh):
    """
    Transfer vertex colors from the original mesh to the
    simplified mesh using nearest-neighbour lookup.

    This is useful when colors represent discrete labels,
    because it avoids interpolating between different labels.
    """

    # Make KD-tree from original vertices
    tree = cKDTree(original_mesh.vertices)

    # Find nearest original vertex for every new vertex
    _, indices = tree.query(simplified_mesh.vertices)

    # Transfer RGBA colors
    simplified_mesh.visual.vertex_colors = (
        original_mesh.visual.vertex_colors[indices]
    )

    return simplified_mesh


# ============================================================
# MESH DOWNSAMPLING
# ============================================================

def downsample_mesh(mesh, target_faces):
    """
    Downsample a Trimesh using quadric-error decimation.

    If the mesh has vertex colors, they are transferred to
    the simplified mesh using nearest-neighbour lookup.

    Parameters
    ----------
    mesh : trimesh.Trimesh
        Input mesh.

    target_faces : int
        Desired number of faces.

    Returns
    -------
    trimesh.Trimesh
        Simplified mesh.
    """

    print("Original mesh:")
    print(f"  Vertices: {len(mesh.vertices):,}")
    print(f"  Faces:    {len(mesh.faces):,}")

    # --------------------------------------------------------
    # Check whether vertex colors exist
    # --------------------------------------------------------

    has_vertex_colors = (
        hasattr(mesh.visual, "vertex_colors")
        and mesh.visual.vertex_colors is not None
        and len(mesh.visual.vertex_colors) == len(mesh.vertices)
    )

    if has_vertex_colors:
        print("  Vertex colors: yes")
    else:
        print("  Vertex colors: no")

    # --------------------------------------------------------
    # Decimate
    # --------------------------------------------------------

    simplified = mesh.simplify_quadric_decimation(
        face_count=target_faces
    )

    # --------------------------------------------------------
    # Transfer colors if present
    # --------------------------------------------------------

    if has_vertex_colors:
        simplified = transfer_vertex_colors(
            mesh,
            simplified
        )

    # --------------------------------------------------------
    # Print result
    # --------------------------------------------------------

    print("\nSimplified mesh:")
    print(f"  Vertices: {len(simplified.vertices):,}")
    print(f"  Faces:    {len(simplified.faces):,}")

    return simplified

# With color example

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

INPUT_MESH = "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Master_mesh/brain_segmented_00.obj"
OUTPUT_MESH = "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/brain_segmented_00_downsampled.obj"

TARGET_FACES = 50_000



# ============================================================
# LOAD
# ============================================================

mesh = trimesh.load(
    INPUT_MESH,
    process=False
)


# ============================================================
# DOWNSAMPLE
# ============================================================

simplified_mesh = downsample_mesh(
    mesh,
    target_faces=TARGET_FACES
)


# ============================================================
# SAVE
# ============================================================

simplified_mesh.export(OUTPUT_MESH)

print(f"\nSaved to:")
print(OUTPUT_MESH)

Original mesh:
  Vertices: 225,185
  Faces:    450,784
  Vertex colors: yes

Simplified mesh:
  Vertices: 24,776
  Faces:    50,000

Saved to:
/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/brain_segmented_00_downsampled.obj


# without color example

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

INPUT_MESH = "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Master_mesh/brain_segmented_00.obj"
OUTPUT_MESH = "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/brain_segmented_00_downsampled.obj"

TARGET_FACES = 100_000



# ============================================================
# LOAD
# ============================================================

mesh = trimesh.load(
    INPUT_MESH,
    process=False
)

print(f"  Original: {len(mesh.vertices):,} vertices, "
          f"{len(mesh.faces):,} faces")

# ============================================================
# DOWNSAMPLE
# ============================================================

simplified_mesh = mesh.simplify_quadric_decimation(
            face_count=TARGET_FACES
        )

print(f"  Result:   {len(simplified_mesh.vertices):,} vertices, "
          f"{len(simplified_mesh.faces):,} faces")

# ============================================================
# SAVE
# ============================================================

simplified_mesh.export(OUTPUT_MESH)

print(f"\nSaved to:")
print(OUTPUT_MESH)

  Original: 225,185 vertices, 450,784 faces
  Result:   24,776 vertices, 50,000 faces

Saved to:
/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/brain_segmented_00_downsampled.obj


# Directories

In [3]:
import csv
from pathlib import Path

import numpy as np
import trimesh
from scipy.spatial import cKDTree

# ============================================================
# SETTINGS
# ============================================================

INPUT_DIR = Path("/Users/jacob/Desktop/Uni/7_semester/Bachelor/Meshes")
OUTPUT_DIR = Path(
    "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes"
)

# Target number of faces
TARGET_FACES = 100_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_obj_vertex_colors(obj_path):
    """Return normalized RGB values from OBJ `v x y z r g b` lines, or None."""
    vertex_colors = []
    has_color_flags = []

    with obj_path.open("r", encoding="utf-8", errors="replace") as obj_file:
        for line in obj_file:
            fields = line.strip().split()
            if not fields or fields[0] != "v":
                continue

            values = [float(value) for value in fields[1:]]

            if len(values) >= 6:
                vertex_colors.append(values[3:6])
                has_color_flags.append(True)
            elif len(values) == 3:
                vertex_colors.append(None)
                has_color_flags.append(False)
            else:
                raise ValueError(
                    f"Unexpected vertex format in {obj_path.name}: {line.strip()}"
                )

    if not any(has_color_flags):
        return None

    if not all(has_color_flags):
        raise ValueError(
            f"{obj_path.name} mixes colored and uncolored vertex lines; "
            "all vertices must have either XYZ or XYZRGB."
        )

    colors = np.asarray(vertex_colors, dtype=np.float64)

    # OBJ vertex colors are commonly 0..1; also accept 0..255 values.
    if colors.max(initial=0.0) > 1.0:
        colors /= 255.0

    return np.clip(colors, 0.0, 1.0)


def write_rgb_csv(csv_path, rgb_0_to_1):
    """Write normalized RGB values, one row per output vertex."""
    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["red", "green", "blue"])
        writer.writerows(rgb_0_to_1.tolist())


def clean_mesh(mesh):
    """Remove degenerate and duplicate faces and unused vertices."""
    mesh.update_faces(mesh.nondegenerate_faces())
    mesh.update_faces(mesh.unique_faces())
    mesh.remove_unreferenced_vertices()

    if len(mesh.faces) == 0:
        raise ValueError("No valid faces remain after decimation cleanup.")

    if not np.isfinite(mesh.vertices).all():
        raise ValueError("Mesh contains non-finite vertex coordinates.")

    if np.any(mesh.area_faces <= 0):
        raise ValueError("Mesh still contains zero-area faces after cleanup.")

    return mesh


# ============================================================
# PROCESS ALL MESHES
# ============================================================

for mesh_path in INPUT_DIR.glob("*.obj"):
    print(f"Processing: {mesh_path.name}")

    try:
        source_colors = read_obj_vertex_colors(mesh_path)
        mesh = trimesh.load(mesh_path, process=False, force="mesh")

        if not isinstance(mesh, trimesh.Trimesh):
            print(f"  Skipping {mesh_path.name}: not a single mesh")
            continue

        if source_colors is not None and len(source_colors) != len(mesh.vertices):
            raise ValueError(
                "OBJ vertex color count does not match Trimesh vertex count "
                f"({len(source_colors)} vs {len(mesh.vertices)})."
            )

        print(
            f"  Original: {len(mesh.vertices):,} vertices, "
            f"{len(mesh.faces):,} faces"
        )
        print(
            f"  Vertex colors: "
            f"{'found' if source_colors is not None else 'none'}"
        )

        # Keep original coordinates for transferring colors to output vertices.
        original_vertices = np.asarray(mesh.vertices).copy()

        if len(mesh.faces) <= TARGET_FACES:
            simplified_mesh = mesh.copy()
            print("  Already below target, no simplification needed")
        else:
            simplified_mesh = mesh.simplify_quadric_decimation(
                face_count=TARGET_FACES
            )

        # Clean up invalid faces before assigning colors or exporting.
        simplified_mesh = clean_mesh(simplified_mesh)

        print(
            f"  Result:   {len(simplified_mesh.vertices):,} vertices, "
            f"{len(simplified_mesh.faces):,} faces"
        )

        output_colors = None
        if source_colors is not None:
            # Assign each final vertex the color of its nearest original vertex.
            nearest_original = cKDTree(original_vertices).query(
                simplified_mesh.vertices, k=1
            )[1]
            output_colors = source_colors[nearest_original]

            simplified_mesh.visual.vertex_colors = np.rint(
                np.clip(output_colors, 0.0, 1.0) * 255
            ).astype(np.uint8)

        # Export once, after cleanup and color assignment.
        output_path = OUTPUT_DIR / mesh_path.name
        simplified_mesh.export(output_path)
        print(f"  Saved: {output_path}")

        if output_colors is not None:
            colors_csv_path = OUTPUT_DIR / f"{mesh_path.stem}_colors.csv"
            write_rgb_csv(colors_csv_path, output_colors)
            print(f"  Saved RGB CSV: {colors_csv_path}")

    except Exception as exc:
        print(f"  Failed {mesh_path.name}: {exc}")

    print()

print("Done!")

Processing: final_112_A_RHOD.obj
  Original: 420,554 vertices, 841,092 faces
  Vertex colors: none
  Result:   49,989 vertices, 99,965 faces
  Saved: /Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/final_112_A_RHOD.obj

Processing: final_063_B_ISET.obj
  Original: 320,882 vertices, 641,748 faces
  Vertex colors: none
  Result:   49,993 vertices, 99,969 faces
  Saved: /Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/final_063_B_ISET.obj

Processing: final_140_B_WEIM.obj
  Original: 350,074 vertices, 700,148 faces
  Vertex colors: none
  Result:   49,990 vertices, 99,984 faces
  Saved: /Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/final_140_B_WEIM.obj

Processing: final_140_C_WEIM.obj
  Original: 340,884 vertices, 681,752 faces
  Vertex colors: none
  Result:   49,996 vertices, 99,973 faces
  Saved: /Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes/final_140_C_WEIM.obj

Processing: final_005_B_GREY.obj
  Original: 357,462 ver